# A diffusion model as a browser agent

> An offline replica of laya-ultrafast on Mind2Web, with one model for both of its halves: a masked diffusion model decides each step's operation and element through a LoRA adapter, and, with the adapter off, writes what a typing step types. Recorded steps from websites it never trained on, scored step by step.

- skip_exec: true
- skip_showdoc: true

[laya-ultrafast](https://github.com/cklxx/laya) drives a browser with two models. A small encoder decides each step: which operation comes next (click, type, select, or a control such as *done*) and on which element of the page. When the operation is typing, a small language model writes the text. The decider is fast because it never writes: one forward pass gives a probability to every option.

A [masked diffusion language model](../24_mdlm.ipynb) can be both. It reads the whole text at once, in both directions, as an encoder does, so sysone's head reads a decision off it in one pass: the model's own *Yes* against its *No* at a mask placed after each option. And it writes, by filling a row of masks a few tokens at a time. Here `dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1` does both jobs. A LoRA adapter, trained on Mind2Web steps, makes the decisions; with the adapter switched off the weights are the published model's again, and they write.

The test is [Multimodal-Mind2Web](https://huggingface.co/datasets/osunlp/Multimodal-Mind2Web)'s `test_website` split (OpenRAIL): recorded steps of people doing tasks on websites that none of the training steps came from. There is no browser here. The agent's loop runs over the recordings instead: at each step it decides, writes when it types, and its step is compared with what the person did.

## How the model decides and writes

```{mermaid}
flowchart LR
    S["a recorded step:<br/>the goal, the page, the history"] --> D["decide the operation and the element:<br/>the adapter on, one forward pass"]
    D -- "CLICK or SELECT" --> C["compare with<br/>the person's step"]
    D -- "TYPE_TEXT" --> W["write the text:<br/>the adapter off, 8 unmasking passes"] --> C
```

The decision is read off the model, not written by it. Each option ends with a mask, under the instruction *mark Yes if it answers the question, otherwise No*, and one forward pass gives the model's logits for the tokens *Yes* and *No* at every mask. An option's score is its Yes minus its No, and the softmax across a question's options makes the scores probabilities (the [diffusion model's notebook](../24_mdlm.ipynb#reading-a-decision-yes-against-no) explains why that is a linear head on the model). Writing starts from a row of masks and fills a few at each pass, the most confident first.

![A jev row's masks, the hidden state at each, its product with E[Yes] − E[No], and the softmax across the options; a yes/no question's one mask and σ(s)](../images/mdlm_yesno.svg)

::: {.callout-tip title="Jargon"}
- **Step, operation, element**: one action of a recorded task. The operation is what to do (CLICK, TYPE_TEXT or SELECT, or a control: WAIT, SCROLL_DOWN, SCROLL_UP, DONE, BLOCKED), and the element is where to do it, one of up to 45 candidates from the page.
- **System One, System Two**: the fast decision (one forward pass, a probability for each option) and the slow writing (one forward pass for every unmasking step).
- **Adapter (LoRA)**: a few million weights added beside a model's layers and trained while the model's own weights stay as they are. Switched off, the model is the original one exactly.
- **Exact match**: the text written is the text the person typed, ignoring case, spacing and surrounding quotes.
:::

::: {.callout-note title="Running this notebook"}
The test suite skips this notebook (`skip_exec`). It needs a GPU: it ran as a [Kaggle job](../40_cloud.ipynb) on a T4, `remote("11_mdlm_browser.ipynb", on="kaggle", hw="t4")`, and the outputs below come from that run. It downloads the model (1.5 GB), the adapter (40 MB) and the text columns of 300 `test_website` steps and 1,500 `train` steps. The outputs come from a run on 2026-10-06 with the first adapter, at `revision=FIRST`: the repo has held a better one since, and the last section has its scores.
:::

In [ ]:
import re, time
import numpy as np
import pandas as pd
import torch, transformers
import datasets
from sysone.datasets import from_mm_mind2web
from sysone.cloud import job_input
from sysone.inference import Decider
from sysone.mdlm import ENCODER, zero_shot

In [ ]:
transformers.utils.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar(); datasets.disable_progress_bars()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 70)
ADAPTER = job_input("adapter") or "sgaseretto/diffcider-browser"      # the adapter a job was given, or the published one
FIRST = "b71cab6fe8b72faf6b2710fc3a8fe272af481d57"                   # the repo's revision with the first adapter, the one this notebook ran
device = "cuda" if torch.cuda.is_available() else "cpu"
t_start = time.time()
def rate(x) -> float: return float(np.mean(x)) if len(x) else np.nan        # a share, or NaN when there is nothing to count
print(f"torch {torch.__version__} · transformers {transformers.__version__} · {torch.cuda.get_device_name(0) if device == 'cuda' else device}")

torch 2.11.0+cu128 · transformers 5.16.1 · Tesla T4


## The steps

`from_mm_mind2web` reads Multimodal-Mind2Web shard by shard and makes a sysone record of each recorded action; with `screenshots=False` it reads the text columns only, a small part of the 13.6 GB set. A record asks two questions: `operation`, which operation comes next, and a target question for the operation the person used, asking which element it acts on. A typing step also keeps what was typed, into which field, and whether the task's own wording contains it.

In [ ]:
steps = from_mm_mind2web("test_website", n=300, screenshots=False)
s = next(s for s in steps if s["typed"] and s["typed"]["in_goal"])
q = s["questions"]
target = next(k for k in q if k.endswith("_target"))
print("goal:     ", q["operation"]["instructions"]["goal"])
print("operation:", " · ".join(q["operation"]["criteria"]))
print(f"{target}: {len(q[target]['criteria'])} elements, such as", " · ".join(list(q[target]["criteria"].values())[:3]))
print("the person:", s["gold"]["operation"]["label"], s["gold"][target]["label"], "and typed", repr(s["typed"]["value"]))

goal:      What are the romantic reggae musics from BCD Studio that can be used in tik tok series in andorra
operation: CLICK · TYPE_TEXT · WAIT · SCROLL_DOWN · SCROLL_UP · DONE · BLOCKED
type_text_target: 1 elements, such as [26] Input artist name (textbox)
the person: TYPE_TEXT 26 and typed 'BCD Studio'


The 300 steps come from tasks on 9 websites. Most steps are clicks, and a recorded step is never a control: the people always did something.

In [ ]:
def target_q(s): return next(k for k in s["questions"] if k.endswith("_target"))
pd.DataFrame({"steps": pd.Series([s["gold"]["operation"]["label"] for s in steps]).value_counts(),
              "elements per step": pd.Series({op: rate([len(s["questions"][target_q(s)]["criteria"]) for s in steps
                                                         if s["gold"]["operation"]["label"] == op])
                                              for op in ("CLICK", "TYPE_TEXT", "SELECT")}).round(1)})

,steps,elements per step
CLICK,235,44.6
TYPE_TEXT,43,1.2
SELECT,22,8.0


A typing step offers 1.2 fields on average, so its element is nearly given; a click step chooses among 45.

## The loop

An agent's step is one call to decide and, when it types, one call to write. `predict` answers both questions of a step in one batch, the operation and the element; `generate` writes the text, with the decider's adapter switched off (it does so unless asked otherwise). The text it writes is asked for from the goal, the page and the field the decider chose:

In [ ]:
SYSTEM = "You fill in web forms for a user. Answer with the exact text to type, nothing else."

def ask(step, field:str) -> str:
    "What the writer is asked: the goal, the page and the field"
    return (f"The user's goal: {step['questions']['operation']['instructions']['goal']}\n"
            f"The page: {step['state']['page']['title']}\nThe field: {field}\nWhat should be typed into this field?")

def norm(t) -> str: return " ".join(re.sub(r"^[\s\"'`.:]+|[\s\"'`.:]+$", "", str(t)).casefold().split())

def pick(probs:dict, prior:dict|None=None) -> str:
    "The likeliest operation; with `prior`, the likeliest once each one's share of the training steps is divided out (one they never had is never picked)"
    if prior is None: return max(probs, key=probs.get)
    return max((k for k in probs if prior.get(k)), key=lambda k: probs[k] / prior[k])

def agent(dec, steps, write_steps:int=8, prior:dict|None=None) -> pd.DataFrame:
    "Each recorded step: decide the operation and the element, write when the operation is TYPE_TEXT, and compare with the person's step"
    out = []
    for s in steps:
        q, tq = s["questions"], target_q(s)
        t0 = time.perf_counter(); ans = dec.predict(s["state"], q); ms = 1000 * (time.perf_counter() - t0)
        op, el = pick(ans["operation"]["probabilities"], prior), ans[tq]["choice"]
        row = {"gold": s["gold"]["operation"]["label"], "picked": op, "operation": op == s["gold"]["operation"]["label"],
               "element": el == s["gold"][tq]["label"], "decide ms": ms,
               "types": s["typed"] is not None, "wrote": None, "text": None, "write ms": np.nan}
        if op == "TYPE_TEXT":
            field = re.sub(r"^\[[^\]]*\]\s*", "", q[tq]["criteria"].get(el, ""))       # the element as the decider saw it
            t0 = time.perf_counter()
            row["wrote"] = dec.generate(ask(s, field), max_new_tokens=16, steps=write_steps, block_size=16, system=SYSTEM)
            row["write ms"] = 1000 * (time.perf_counter() - t0)
            if s["typed"]: row["text"] = norm(row["wrote"]) == norm(s["typed"]["value"])
        row["step"] = row["operation"] and row["element"] and (not row["types"] or row["text"] is True)
        out.append(row)
    return pd.DataFrame(out)

def by_operation(df:pd.DataFrame) -> pd.DataFrame:
    "Per operation the person performed: how many steps, how often the operation and the element are right; and how often each operation is picked"
    t = df.groupby("gold").agg(steps=("operation", "size"), operation=("operation", "mean"), element=("element", "mean"))
    return t.join(df["picked"].value_counts().rename("picked"), how="outer").fillna(0).astype({"steps": int, "picked": int}).round(3)

def summary(df:pd.DataFrame) -> dict:
    "Accuracy of the operation, of the element, of the text typed into the right field, and of the whole step; the median times"
    typed = df[df["types"] & df["operation"] & df["element"]]
    return {"operation": df["operation"].mean(), "element": df["element"].mean(), "text, right field": rate(typed["text"].astype(float)),
            "step": df["step"].mean(), "decide ms": df["decide ms"].median(), "write ms": df["write ms"].median()}

## Untrained

`zero_shot` is the model as published, its own Yes against its No at each option's slot and nothing trained. On the steps:

In [ ]:
zs = zero_shot(ENCODER, device=device)
runs = {"untrained": agent(zs, steps)}
pd.Series(summary(runs["untrained"])).round(3)

operation              0.023
element                0.247
text, right field        NaN
step                   0.000
decide ms            409.866
write ms                 NaN
dtype: float64

The operations it picks:

In [ ]:
by_operation(runs["untrained"])

,steps,operation,element,picked
CLICK,235,0.03,0.115,12
DONE,0,0.00,0.000,288
SELECT,22,0.00,0.500,0
TYPE_TEXT,43,0.00,0.837,0


Untrained, the model answers DONE on 288 of the 300 steps: it puts its Yes on *Every requirement is visibly satisfied* before any action, and no recorded step is DONE. Its element is right a quarter of the time: 0.115 on the clicks, where 45 elements compete, and 0.84 on the typing steps, which offer 1.2 fields.

## Writing, by itself

Writing needs no adapter, so the untrained model's writing is all there is to measure: on every typing step, given the field the person typed into, two ways.

- `generate` answers in the model's chat format. The fewer unmasking steps, the faster and the rougher the text: with `steps=16` each of the 16 tokens gets its own pass, with `steps=4` four are kept at each pass.
- `fill` writes into the slots of a fixed text and leaves the rest as it is, here a JSON string, `{"text": "{}"}`: the model sees the JSON around the slot, a way to get JSON without asking for it. A slot is 16 masks, all filled, so the text ends at the string's closing quote.

In [ ]:
typing = [s for s in steps if s["typed"]]
in_goal = np.array([s["typed"]["in_goal"] for s in typing])

def measure(write) -> tuple:
    "How often `write(step)` types what the person typed, overall and by whether the goal says it, its median time, and what it wrote"
    said, ms = [], []
    for s in typing:
        t0 = time.perf_counter(); said.append(write(s)); ms.append(1000 * (time.perf_counter() - t0))
    right = np.array([norm(a) == norm(s["typed"]["value"]) for a, s in zip(said, typing)])
    return {"exact": rate(right), "exact, value in the goal": rate(right[in_goal]), "exact, value not in the goal": rate(right[~in_goal]),
            "ms": np.median(ms)}, said

rows, written = {}, {}
for n in (16, 8, 4):
    rows[f"generate, steps={n}"], written[n] = measure(lambda s: zs.generate(ask(s, s["typed"]["field"]), max_new_tokens=16, steps=n,
                                                                           block_size=16, system=SYSTEM))
rows["fill, a JSON string"], written["fill"] = measure(lambda s: zs.fill(ask(s, s["typed"]["field"]), '{"text": "{}"}', n=16, system=SYSTEM)[0])
print(f"{len(typing)} typing steps, {int(in_goal.sum())} of them typing words of the goal")
pd.DataFrame(rows).T.round(3)

43 typing steps, 37 of them typing words of the goal


,exact,"exact, value in the goal","exact, value not in the goal",ms
"generate, steps=16",0.349,0.405,0.0,752.802
"generate, steps=8",0.326,0.378,0.0,382.304
"generate, steps=4",0.302,0.351,0.0,197.371
"fill, a JSON string",0.209,0.243,0.0,753.302


In [ ]:
pd.DataFrame({"field": [s["typed"]["field"] for s in typing], "typed": [s["typed"]["value"] for s in typing],
              "generate, steps=8": written[8], "fill": written["fill"]}).head(12)

,field,typed,"generate, steps=8",fill
0,Input artist name,BCD Studio,BDC Studio,Romantic reggae music from BCD Studio for TikTok series in Andorra
1,Search,Devin Booker,Devin Booker,Devin Booker jersey
2,"Enter text to search for campgrounds, tours, or other recreation a...",Brooks Camp,"Brooks Camp, Katmai National Park, May 22","Check permit availability for Brooks Camp, Katmai National Park on..."
3,Entry Date,05/22/2023,May 22,2023-05-22T12:00
4,Number of Peoples,4,4,Number of Peoples
5,"Search by name, location or state",Yosemite,Yosemite National Park,Private Vehicle Pass for Yosemite National Park
6,input,94587,Yes,"Private Vehicle Pass for Yosemite National Park, Zip Code 94587"
7,input,12345,Yes,"Private Vehicle Pass for Yosemite National Park, Zip Code 94587"
8,Search,Johannesburg,Johburg vegan,restaurants in Johannesburg
9,What are you looking for?,Google Pixel 7,snow cheapest snow Pixel 7 128gb storage,Find the cheapest snow colored Google Pixel 7 with 128gb storage


On the 43 typing steps of these websites, the model types exactly what the person typed 35% of the time with 16 unmasking steps (753 ms on the T4) and 30% with 4 (197 ms): 41% when the goal's wording contains the value, and never when it does not, since a zip code, or a date in another format, can't be read off the goal. Its misses are mostly longer than the person's text: the park with its name (*Yosemite National Park* for *Yosemite*), the search with its filters (*Nest Hello Video Doorbell 1 star reviews*). `fill` comes out lower, 21%, though its slot now ends at the closing quote: inside the JSON string it writes the goal's whole phrase (*Romantic reggae music from BCD Studio…*), where `generate` answers with the name.

## Trained

The adapter this notebook runs was trained as in the [diffusion model's notebook](../24_mdlm.ipynb#browser-steps-on-the-real-checkpoint): `decision_learner` on 1,500 steps of the `train` split, keeping their websites apart for validation and calibration (`groups="website"`). It was the first one. The adapter published now trained the same way with two changes, every action offered on every step and the operations balanced, and that is what the cell shows (the last section says why). On a laptop, the same few lines go to a Kaggle T4 as a script with `remote("train_browser.py", on="kaggle", hw="t4")`:

In [ ]:
#| eval: false
from sysone.data import TypedDecisions, RowSampler
from sysone.mdlm import decision_learner
train = from_mm_mind2web("train", n=1500, screenshots=False, operations="all")          # every action offered on every step
data = TypedDecisions.from_records(train, valid=0.05, calib=0.1, groups="website", sampler=RowSampler("oversample"))
learn = decision_learner(data, per_device_train_batch_size=4, gradient_accumulation_steps=4)
learn.fit_one_cycle(1, max_steps=150, eval=False)
learn.calibrate()
learn.export("diffcider-browser")         # the adapter alone, and the model's id and commit

`Decider.load` takes the export, from the Hub or from a folder, downloads the model it names, and puts the adapter on it; `revision=FIRST` takes the first adapter from the repo's history:

In [ ]:
del zs
if device == "cuda": torch.cuda.empty_cache()
dec = Decider.load(ADAPTER, revision=FIRST, device=device)
runs["diffcider-browser"] = agent(dec, steps)
pd.Series(summary(runs["diffcider-browser"])).round(3)

operation              0.783
element                0.643
text, right field        NaN
step                   0.457
decide ms            522.593
write ms                 NaN
dtype: float64

Per operation the person performed:

In [ ]:
by_operation(runs["diffcider-browser"])

,steps,operation,element,picked
CLICK,235,1.0,0.583,300
SELECT,22,0.0,0.818,0
TYPE_TEXT,43,0.0,0.884,0


The first adapter answers CLICK on every step. 78% of these steps are clicks, and 85% of the training steps; trained on them as they come, it learned that the answer is nearly always CLICK. Its operation is right exactly as often as always clicking, and it never types, so the loop never writes. The element is where it learned: 0.643, and 0.583 on the clicks, among 45 elements.

## Dividing out the shares

A decision model learns how often each answer is right as well as when. Its probability for an operation is, by Bayes' rule, proportional to how well the step fits the operation times the operation's share of the training steps; dividing the share out leaves the first factor, a decision that no longer leans on how common an operation is. The shares come from the operations of the 1,500 training steps, and an operation they never had, a control, is never picked:

In [ ]:
shares = pd.Series([s["gold"]["operation"]["label"] for s in from_mm_mind2web("train", n=1500, screenshots=False)]).value_counts(normalize=True)
print(shares.round(3).to_dict())
runs["diffcider-browser, shares divided out"] = agent(dec, steps, prior=shares.to_dict())
by_operation(runs["diffcider-browser, shares divided out"])

{'CLICK': 0.847, 'TYPE_TEXT': 0.099, 'SELECT': 0.055}


,steps,operation,element,picked
CLICK,235,0.749,0.583,176
SELECT,22,1.000,0.818,31
TYPE_TEXT,43,0.953,0.884,93


With the shares divided out it types on 93 steps and selects on 31, and gets 41 of the 43 typing steps and all 22 select steps; but it now types or selects on 59 click steps too, and the operation as a whole is right 79.7% of the time. That is exactly what the rule *the rarest action offered* gets. A step offers the actions its candidate elements allow, its own element always among them, so every typing step offers TYPE_TEXT and every select step SELECT, while 176 of these 300 steps offer CLICK alone: divided out, the adapter follows the actions on offer (the [screens](../21_screens.ipynb#one-step-one-record) notebook has the finding).

When it types, the same decider writes with its adapter off, so what it typed is what the untrained model writes:

In [ ]:
done = runs["diffcider-browser, shares divided out"]
done[done["wrote"].notna()][["gold", "element", "wrote", "text"]].head(10)

,gold,element,wrote,text
3,CLICK,True,BCD Studio,None
5,TYPE_TEXT,True,BDC Studio,False
6,CLICK,True,Romantic reggae music,None
8,CLICK,False,Trending songs,None
9,CLICK,False,Canada,None
14,CLICK,True,Rock playlist,None
15,CLICK,False,Rock,None
22,TYPE_TEXT,True,Devin Booker,True
26,CLICK,True,Add to Cart,None
38,TYPE_TEXT,True,Katmai National Park,False


Where it types into the right field, the text is right on 31% of the steps (*text, right field* below), as writing by itself measured.

## Results

Step by step on the 300 steps:

In [ ]:
clicks = rate([s["gold"]["operation"]["label"] == "CLICK" for s in steps])
pd.DataFrame({"always CLICK": {"operation": clicks}} | {k: summary(v) for k, v in runs.items()}).T.round(3)

,operation,element,"text, right field",step,decide ms,write ms
always CLICK,0.783,NaN,NaN,NaN,NaN,NaN
untrained,0.023,0.247,NaN,0.000,409.866,NaN
diffcider-browser,0.783,0.643,NaN,0.457,522.593,NaN
"diffcider-browser, shares divided out",0.797,0.643,0.306,0.447,520.765,485.217


For scale, [laya-ultrafast's write-up](https://github.com/cklxx/laya/blob/main/docs/finetune_browser_agent.md) reports, for its fine-tuned encoders, the right operation 88 to 89% of the time (54% untrained) and the right element 63% (322M parameters) and 66% (421M) of the time, against 10% untrained, in 17 to 23 ms and 41 to 50 ms a step. Those numbers are not on these steps: it also scored held-out Mind2Web pages with about 45 candidates each, but it trained on Mind2Web together with 5,244 generated goals, 700 real DONE pages and on-policy corrections, and its operations include DONE.

## A better adapter

Two changes make an adapter that decides the operation: every action offered on every step while it trains (`from_mm_mind2web(..., operations="all")`), so that the offered actions never give the answer away, and the operations balanced (`RowSampler("oversample")`), so that clicking is not the safe answer. The [diffusion model's notebook](../24_mdlm.ipynb#every-action-offered) trains it and three other variants and scores them all on these 300 steps; the published `sgaseretto/diffcider-browser` holds it, and `Decider.load(ADAPTER)`, without a revision, runs it. On these steps:

| | operation | element | step |
|---|---|---|---|
| always CLICK | 0.783 | | |
| the rarest action offered | 0.797 | | |
| the first adapter (this notebook) | 0.783 | 0.643 | 0.457 |
| the published adapter | **0.853** | **0.647** | **0.470** |

It picks the operation from the goal, the page and the steps before, and beats the rule: CLICK on 96.6% of the click steps, TYPE_TEXT on 42% of the typing steps, SELECT on half the select steps. Offered every action on every step, it leans toward typing and selecting (57% right), having trained on the operations in equal shares: offer it the actions the page's elements allow.

In [ ]:
print(f"the whole notebook: {(time.time() - t_start) / 60:.1f} minutes")

the whole notebook: 13.6 minutes
